# Predicción de accidentalidad — El Poblado

Punto de partida para la parte de leaderboard. **A propósito no trae el flujo resuelto**: la construcción del target, la limpieza, la partición de validación y la ingeniería de características son parte del trabajo evaluado (secciones 4.1 a 4.4 del taller).

Lo único que se fija aquí es lo que no es negociable: cómo conectarse y en qué formato exacto debe quedar la entrega.

Lean el README del assignment antes de empezar.

## 1. Conexión a los datos

In [ ]:
import sqlite3

import pandas as pd

DB_PATH = "data_accidentes_poblado.sqlite3"

con = sqlite3.connect(DB_PATH)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
# Exploren cada tabla: qué representa una fila, qué llaves las cruzan,
# qué rango de fechas cubre cada una. Hay al menos una inconsistencia
# entre tablas que vale la pena que encuentren (sección 4.2 del taller).
con.close()

## 2. Su trabajo

En su propio notebook o a partir de aquí:

- EDA y calidad de datos
- Definición y construcción de la variable objetivo
- Unión de tablas e ingeniería de características
- Estrategia de validación (¿por qué una partición aleatoria sería un error aquí?)
- Manejo del desbalance, comparación de modelos, selección final

El README explica la regla que deben respetar al construir variables.

## 3. Formato de la entrega

Esto sí es fijo. El archivo debe tener exactamente las columnas `id` y `target`, y una fila por cada pareja (barrio, hora) del período de evaluación.

- `id`: `"{BARRIO}|{TW}"`, con `TW` como `YYYY-MM-DD HH:MM:SS`
- `target`: probabilidad continua entre 0 y 1

In [ ]:
def construir_submission(barrios, marcas_tiempo, probabilidades, ruta="mi_prediccion.csv"):
    """Arma el CSV de entrega en el formato que espera el leaderboard."""
    sub = pd.DataFrame({
        "id": pd.Series(barrios).astype(str)
        + "|"
        + pd.to_datetime(pd.Series(marcas_tiempo)).dt.strftime("%Y-%m-%d %H:%M:%S"),
        "target": probabilidades,
    })
    if len(sub) != 80256:
        raise ValueError(f"Se esperaban 80256 filas, hay {len(sub)}")
    if sub["id"].duplicated().any():
        raise ValueError("Hay ids duplicados")
    if not sub["target"].between(0, 1).all():
        raise ValueError("target debe estar entre 0 y 1")
    sub.to_csv(ruta, index=False)
    return sub


# construir_submission(test["BARRIO"], test["TW"], mis_probabilidades)